# Strategy Development Laboratory

Copy this notebook, rename it (e.g. `lighter_c_bbo_v2.ipynb`), and develop your strategy.

**Workflow:** Load data → Define strategy → Test

The strategy name is automatically derived from the notebook name. When you edit the strategy cell and run it, the file updates automatically.

In [ ]:
import json
from pathlib import Path
import os
import numpy as np
import pandas as pd
from leadlag import list_sessions, load_session, Strategy, Order, Context, Event, BboSnapshot

DATA = '../data'
sessions = list_sessions(DATA)
session = load_session(sessions[0]['id'], data_dir=DATA)
events = session.events.filter(signal='C')

print(f"Session: {session.session_id}")
print(f"Duration: {session.meta['duration_s']:.0f}s  |  Events: {session.events.count}  |  Signal C: {events.count}")
print(f"Followers: {session.meta['followers']}")
fees_preview = ', '.join(
    f"{v}: {session.meta['fees'][v]['taker_bps']}"
    for v in session.meta['followers'][:3]
)
print(f"Fees (taker bps): {fees_preview}...")
bbo_available = [v for v, ok in session.meta.get('bbo_available', {}).items() if ok][:5]
print(f"BBO available: {bbo_available}...")

In [ ]:
def get_notebook_name():
    """Get current notebook name from Jupyter kernel session"""
    runtime_dir = Path(os.environ.get('JUPYTER_RUNTIME_DIR', '~/.jupyter/runtime')).expanduser()
    for session_file in runtime_dir.glob('kernel-*.json'):
        try:
            session_data = json.loads(session_file.read_text())
            jupyter_session = session_data.get('jupyter_session', '')
            if jupyter_session and '.ipynb' in jupyter_session:
                return Path(jupyter_session).stem
        except:
            pass
    return "strategy"

STRATEGY_NAME = get_notebook_name()
print(f"Strategy will be saved as: {STRATEGY_NAME}")

In [ ]:
%%writefile ../data/strategies/{STRATEGY_NAME}.py
from leadlag import Strategy, Order


class MyStrategy(Strategy):
    version = "2026-04-18"
    description = "Strategy template"
    params = {
        "signal": "C",
        "follower": "Lighter Perp",
        "min_magnitude": 2.0,
        "hold_ms": 30000,
    }

    def on_event(self, event, ctx):
        p = self.params

        if event.signal != p["signal"]:
            return None
        if p["follower"] not in event.lagging_followers:
            return None
        if event.magnitude_sigma < p["min_magnitude"]:
            return None

        return Order(
            venue=p["follower"],
            side="buy" if event.direction > 0 else "sell",
            qty_btc=0.001,
            entry_type="market",
            hold_ms=p["hold_ms"],
        )

In [ ]:
from leadlag import load_strategy

strat = load_strategy(f'../data/strategies/{STRATEGY_NAME}.py')
print(f"Loaded: {strat.name}")
print(f"Params: {strat.params}")

# Quick test with mock event
event = Event(
    bin_idx=0, ts_ms=0, signal='C', direction=1,
    magnitude_sigma=3.0, leader='OKX Perp', lagging_followers=['Lighter Perp']
)
ctx = Context(ts_ms=0, bbo={'Lighter Perp': BboSnapshot('Lighter Perp', spread_bps=1.5)})
order = strat.on_event(event, ctx)
print(f"\nTest order: {order}")

In [ ]:
# Test with params variations
print("Test 1: Change min_magnitude")
strat.params['min_magnitude'] = 5.0
order = strat.on_event(event, ctx)
print(f"  Order (stricter): {order}")

print("\nTest 2: Change follower")
strat.params['follower'] = 'Binance Perp'
event.lagging_followers = ['Binance Perp']
order = strat.on_event(event, ctx)
print(f"  Order (other follower): {order}")

# Reset
strat.params['min_magnitude'] = 2.0
strat.params['follower'] = 'Lighter Perp'
event.lagging_followers = ['Lighter Perp']